In [ ]:
import sys

import os
import re
import click
import tqdm
import pickle
import numpy as np
import torch
import PIL.Image
import dnnlib
from torch_utils import distributed as dist
from PIL import Image
from tqdm import tqdm
import torch.nn.functional as F

from matplotlib import pyplot as plt


In [ ]:
def edm_sampler(
	net, latents, conditions, class_labels=None, randn_like=torch.randn_like,
	num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
	S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
	# Adjust noise levels based on what's supported by the network.
	sigma_min = max(sigma_min, net.sigma_min)
	sigma_max = min(sigma_max, net.sigma_max)

	# Time step discretization.
	step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
	t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
	t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

	# Main sampling loop.
	x_next = latents.to(torch.float64) * t_steps[0]
	for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
		x_cur = x_next

		# Increase noise temporarily.
		gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
		t_hat = net.round_sigma(t_cur + gamma * t_cur)
		x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

		# Euler step.
		denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
		d_cur = (x_hat - denoised) / t_hat
		x_next = x_hat + (t_next - t_hat) * d_cur

		# Apply 2nd order correction.
		if i < num_steps - 1:
			denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
			d_prime = (x_next - denoised) / t_next
			x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

	return x_next



#----------------------------------------------------------------------------
# Wrapper for torch.Generator that allows specifying a different random seed
# for each sample in a minibatch.

class StackedRandomGenerator:
	def __init__(self, device, seeds):
		super().__init__()
		self.generators = [torch.Generator(device).manual_seed(int(seed) % (1 << 32)) for seed in seeds]

	def randn(self, size, **kwargs):
		assert size[0] == len(self.generators)
		return torch.stack([torch.randn(size[1:], generator=gen, **kwargs) for gen in self.generators])

	def randn_like(self, input):
		return self.randn(input.shape, dtype=input.dtype, layout=input.layout, device=input.device)

	def randint(self, *args, size, **kwargs):
		assert size[0] == len(self.generators)
		return torch.stack([torch.randint(*args, size=size[1:], generator=gen, **kwargs) for gen in self.generators])

#----------------------------------------------------------------------------
# Parse a comma separated list of numbers or ranges and return a list of ints.
# Example: '1,2,5-10' returns [1, 2, 5, 6, 7, 8, 9, 10]

def parse_int_list(s):
	if isinstance(s, list): return s
	ranges = []
	range_re = re.compile(r'^(\d+)-(\d+)$')
	for p in s.split(','):
		m = range_re.match(p)
		if m:
			ranges.extend(range(int(m.group(1)), int(m.group(2))+1))
		else:
			ranges.append(int(p))
	return ranges

#----------------------------------------------------------------------------

In [ ]:
# device 
device=torch.device('cuda:0')


network_pkl = '/home/zoz/cond-diff-turb/outputs/00000-SQG_32_lo_res_dynamic-medium-gpus1-batch256-fp32/network-snapshot-002000.pkl'
# Load network.
print(f'Loading network from "{network_pkl}"...')
with dnnlib.util.open_url(network_pkl, verbose=True) as f:
	net_32_lo = pickle.load(f)['ema'].to(device)



network_pkl = '/home/zoz/cond-diff-turb/outputs/00001-SQG_32_hi_patch-medium-gpus1-batch256-fp32/network-snapshot-002000.pkl'
# Load network.
print(f'Loading network from "{network_pkl}"...')
with dnnlib.util.open_url(network_pkl, verbose=True) as f:
	net_32_hi = pickle.load(f)['ema'].to(device)


network_pkl = '/home/zoz/edm/training-32-corp-joint-nv/00004-sqg_32_corp-uncond-z5-cedm-gpus1-batch256-fp32/network-snapshot-002000.pkl'
# Load network.
print(f'Loading network from "{network_pkl}"...')
with dnnlib.util.open_url(network_pkl, verbose=True) as f:
	net_sr = pickle.load(f)['ema'].to(device)

In [ ]:
# arguemnt variblas for the script
# Variables with default values
outdir = 'output'       # Required, so assign path before running
subdirs = False
class_idx = None
max_batch_size = 50
seeds = parse_int_list('0-49')


# batch size
num_batches = ((len(seeds) - 1) // (max_batch_size ) + 1) 
all_batches = torch.as_tensor(seeds).tensor_split(num_batches)
rank_batches = all_batches
batch_seeds = all_batches[0]
batch_size = len(batch_seeds)

# Loop over batches.
print(f'Generating {len(seeds)} images to "{outdir}"...')


# Pick latents and labels.
rnd = StackedRandomGenerator(device, batch_seeds)


In [ ]:
# load some data 


fields_all_np = np.load('../../datasets/SQG/SQG_fields.npy')
print(fields_all_np.shape)
print(fields_all_np[:100,...].min())
print(fields_all_np[:100,...].max())

In [ ]:

from IPython.display import HTML
import matplotlib.animation as animation
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 30 # in MB (30 MB in this example)


n_sample_check = 50

# show sqg
n_frame_show = n_sample_check
fields_all = fields_all_np
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat0.set_data(fields_all[i])
	ax.set_title(f'pv_all_0 (idx = {i}) ')

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 5))
mat0 = ax.imshow(fields_all[n_start], vmax=1, vmin=-1)
fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()


In [ ]:
def gen_grid_weight(block_res, expand_ratio):
	grid_size = expand_ratio*2 - 1
	total_res =  block_res * expand_ratio

	# get 1d basis function template 
	x_vec = torch.linspace(0, np.pi, block_res)
	b_1d_left = (torch.cos(x_vec) + 1) / 2
	b_1d_mid = -(torch.cos(x_vec*2) + 1) / 2 + 1
	b_1d_right = torch.flip(b_1d_left, dims=[0])

	# get 1d basis function grid 
	b_1d_grid = [b_1d_left]
	for i in range(grid_size-2):
		b_1d_grid.append(b_1d_mid)    
	b_1d_grid.append(b_1d_right)
	

	# Create a 2D grid of basis functions by taking outer products of the 1D basis functions
	b_2d_grid = []
	for i, b_1d_x in enumerate(b_1d_grid):
		temp = []
		for j, b_1d_y in enumerate(b_1d_grid):
			b_2d = torch.outer(b_1d_x, b_1d_y)
			temp.append(b_2d)
		b_2d_grid.append(temp)


	# pad into full grid
	b_2d_grid_padded = []
	for i in range(grid_size):
		temp = []
		for j in range(grid_size):
			b_2d_padded = torch.zeros((total_res, total_res))
			loc_x = i*block_res//2
			loc_y = j*block_res//2
			b_2d_padded[loc_x:loc_x+block_res, loc_y:loc_y+block_res] = b_2d_grid[i][j]
			temp.append(b_2d_padded)

		b_2d_grid_padded.append(temp)

	# calculate the total weight for each padded block
	# this is used to normalize the weight of each padded block
	w_total = torch.zeros((total_res, total_res))
	for i in range(grid_size):
		for j in range(grid_size):
			w_total += b_2d_grid_padded[i][j]



	# adjust weight to all padded weight 
	b_2d_grid_padded_adjusted = []
	for i in range(grid_size):
		temp = []
		for j in range(grid_size):
			temp.append(b_2d_grid_padded[i][j] / w_total)
		b_2d_grid_padded_adjusted.append(temp)


	# get back the original block res weight 
	b_2d_grid_adjusted = []
	for i in range(grid_size):
		temp = []
		for j in range(grid_size):
			loc_x = i*block_res//2
			loc_y = j*block_res//2
			temp.append(b_2d_grid_padded_adjusted[i][j][loc_x:loc_x+block_res, loc_y:loc_y+block_res])

		b_2d_grid_adjusted.append(temp)

	b_2d_grid_adjusted = torch.stack([torch.stack(row) for row in b_2d_grid_adjusted])
	return b_2d_grid_adjusted


block_res = 32
expand_ratio = 4
weight_grid_2d = gen_grid_weight(block_res, expand_ratio)
print(weight_grid_2d.shape)
weight_grid_1d = weight_grid_2d.reshape(49, 1, 32, 32)
weight_grid_1d = weight_grid_1d.to(device)
print(weight_grid_1d.shape)




In [ ]:

def apply_denoiser_pou(net, x, t, conditions, class_labels,
	weight_grid: torch.Tensor,
	patch_size: int = 32,
	stride: int = 16
) -> torch.Tensor:
	"""
	Apply a denoiser to each patch and combine with PoU.

	Args:
		images: Tensor of shape (B, C, H, W)
		net: Callable that processes patches: (B*num_patches, C, patch_size, patch_size) -> same shape
		weight_grid: PoU weights (grid_h, grid_w, patch_size, patch_size)
		patch_size: Size of square patches
		stride: Stride for unfolding and folding

	Returns:
		Tensor of shape (B, C, H, W)
	"""
	images = torch.concat([x, conditions], dim=1)

	B, C, H, W = images.shape

	# Step 1: Extract patches
	patches = F.unfold(images, kernel_size=patch_size, stride=stride)  # (B, C*ps*ps, num_patches)
	num_patches = patches.shape[-1]
	grid_h = grid_w = int(num_patches**0.5)
	assert grid_h * grid_w == num_patches, "Image must produce square patch grid"

	# Step 2: Reshape to (B * num_patches, C, patch_size, patch_size)
	patches = patches.transpose(1, 2).reshape(B * num_patches, C, patch_size, patch_size)

	# Step 3: Apply user-defined function
	# processed = net(patches)         # (B * num_patches, C, patch_size, patch_size)
	processed = net(patches[:,[0],:,:], t, patches[:,[1,2],:,:], class_labels)         # (B * num_patches, C, patch_size, patch_size)

	# Step 4: Multiply with shared weight grid
	# weights: (grid_h, grid_w, patch_size, patch_size) -> (num_patches, 1, patch_size, patch_size)
	# weight_grid = weight_grid.reshape(num_patches, 1, patch_size, patch_size)

	# Expand and apply weights
	processed = processed.view(B, num_patches, C, patch_size, patch_size)
	weighted = processed * weight_grid.unsqueeze(0)  # (B, num_patches, C, patch_size, patch_size)
	weighted = weighted.view(B * num_patches, C, patch_size, patch_size)

	# Step 5: Reshape to (B, C * patch_size * patch_size, num_patches)
	weighted = weighted.view(B, num_patches, C * patch_size * patch_size).transpose(1, 2)

	# Step 6: Fold back to image
	reconstructed = F.fold(weighted, output_size=(H, W), kernel_size=patch_size, stride=stride)

	return reconstructed


In [ ]:
def f2p(fields, patch_size: int = 32, stride: int = 16):
	B, C, H, W = fields.shape

	# Step 1: Extract patches
	patches = F.unfold(fields, kernel_size=patch_size, stride=stride)  # (B, C*ps*ps, num_patches)
	num_patches = patches.shape[-1]
	grid_h = grid_w = int(num_patches**0.5)
	assert grid_h * grid_w == num_patches, "Image must produce square patch grid"

	# Step 2: Reshape to (B * num_patches, C, patch_size, patch_size)
	patches = patches.transpose(1, 2).reshape(B * num_patches, C, patch_size, patch_size)
	return patches


def p2f(processed_patches, weight_grid_1d):
	num_patches = weight_grid_1d.shape[0]
	patch_size = weight_grid_1d.shape[2]
	C = 1
	B = processed_patches.shape[0] // num_patches
	H = 128 
	W = 128 
	stride = 16
	# Step 4: Multiply with shared weight grid
	# weights: (grid_h, grid_w, patch_size, patch_size) -> (num_patches, 1, patch_size, patch_size)
	# weight_grid = weight_grid.reshape(num_patches, 1, patch_size, patch_size)

	# Expand and apply weights
	processed_patches = processed_patches.view(B, num_patches, C, patch_size, patch_size)
	weighted = processed_patches * weight_grid_1d.unsqueeze(0)  # (B, num_patches, C, patch_size, patch_size)
	weighted = weighted.view(B * num_patches, C, patch_size, patch_size)

	# Step 5: Reshape to (B, C * patch_size * patch_size, num_patches)
	weighted = weighted.view(B, num_patches, C * patch_size * patch_size).transpose(1, 2)

	# Step 6: Fold back to image
	reconstructed = F.fold(weighted, output_size=(H, W), kernel_size=patch_size, stride=stride)

	return reconstructed


In [ ]:
def edm_sampler_sr(
	net, latents, conditions, class_labels=None, randn_like=torch.randn_like,
	num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
	S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
	# Adjust noise levels based on what's supported by the network.
	sigma_min = max(sigma_min, net.sigma_min)
	sigma_max = min(sigma_max, net.sigma_max)

	# Time step discretization.
	step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
	t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
	t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

	# condition (BCHW)
	# patch_cond: (B*7*7,C,32,32)
	patch_cond = f2p(conditions)


	# Main sampling loop.
	x_next = latents.to(torch.float64) * t_steps[0]
	for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
		x_cur = x_next

		# Increase noise temporarily.
		gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
		t_hat = net.round_sigma(t_cur + gamma * t_cur)
		x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

		# Euler step.
		# denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
		patch_x_hat = f2p(x_hat)
		# patch_denoised = net(patch_x_hat, t_hat, patch_cond, class_labels).to(torch.float64)
		patch_denoised = net(patch_x_hat, patch_cond, t_hat, class_labels).to(torch.float64) # sr only model
		
		# denoised(BCWH) 
		denoised = p2f(patch_denoised, weight_grid_1d)


		d_cur = (x_hat - denoised) / t_hat
		x_next = x_hat + (t_next - t_hat) * d_cur

		# Apply 2nd order correction.
		if i < num_steps - 1:
			# denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
			patch_x_next = f2p(x_next)
			# patch_denoised = net(patch_x_next, t_next, patch_cond, class_labels).to(torch.float64)
			patch_denoised = net(patch_x_next, patch_cond, t_next, class_labels).to(torch.float64) # sr only model
			denoised = p2f(patch_denoised, weight_grid_1d)


			d_prime = (x_next - denoised) / t_next
			x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

	return x_next

In [ ]:
# generat a single path 
batch_seeds = torch.tensor([0], device=device)
rnd = StackedRandomGenerator(device, batch_seeds)
init = torch.tensor(fields_all_np[0], dtype=torch.float32, device=device)
init = init[None, None,:,:]
print(init.shape)

In [ ]:

n_pred_step = 30
cur_hi = init.clone()
field_dm = [cur_hi.clone()]
for i in tqdm(range(n_pred_step), desc="Generating 32x32 turb fields"):
	latents_hi = rnd.randn([1, 1, 128, 128], device=device)
	latents_lo = F.interpolate(latents_hi, size=(32, 32), mode='nearest')


	cur_lo = F.interpolate(cur_hi, size=(32, 32), mode='nearest')

	# # low res pred 
	pred_lo = edm_sampler(net_32_lo, latents_lo, cur_lo)
	pred_hi = F.interpolate(pred_lo, size=(128, 128), mode='nearest')

	# low res (data) 
	# pred_lo = torch.tensor(fields_all_np[i+1], dtype=torch.float32, device=device)[None, None,:,:]
	# pred_hi = F.interpolate(pred_lo, size=(128, 128), mode='nearest')


	# super res (dy) 
	# cond_sr = torch.concat([cur_hi, pred_hi], dim=1)
	# pred_hi = edm_sampler_sr(net_32_hi, latents_hi, cond_sr)
	
	# super res (static) 
	cond_sr = pred_hi
	pred_hi = edm_sampler_sr(net_sr, latents_hi, cond_sr)
	

	cur_hi = pred_hi.clone()

	field_dm.append(cur_hi)

field_dm = torch.stack(field_dm, dim=0).cpu().numpy()
print(field_dm.shape)
print(field_dm.min(), field_dm.max())

In [ ]:
# visualize 
# your existing setup...
n_frame_show = n_pred_step
fields_all = field_dm
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat0.set_data(fields_all[i,0,0,...])
	ax.set_title(f'pv_all_0 (idx = {i}) ')

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 5))
mat0 = ax.imshow(fields_all[n_start,0,0,...], vmax=1, vmin=-1)
fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()


In [ ]:
def rmse(x):
	return np.sqrt(np.mean(x**2, axis=(1,2)))

field1 = fields_all[:, 0, 0,...]
field2 = fields_all[:, 1, 0,...]

In [ ]:
diff = field1 - field2
print(diff.shape)

diff_rmse = rmse(diff) / rmse(field2)

plt.plot(diff_rmse)
plt.show()

In [ ]:
field_diff = field1[1:, ...] - field1[:-1, ...]
print(field_diff.shape)

diff_rmse = rmse(field_diff) / rmse(field1[:-1])

plt.plot(diff_rmse)
plt.ylim(0, 1)
plt.show()

In [ ]:
# tau  = 1 
field_corr_gen = []
for tau in range(1, 50):
	field_corr_gen.append(np.mean(field1[tau:, ...] * field1[:-tau, ...]) )

plt.plot(field_corr_gen)
plt.show()

In [ ]:
n_sample_check = 500
feild_32 = []
with zipfile.ZipFile('/home/zoz/EDM-cond-turb/data/SQG/SQG_32_dynamic.zip', 'r') as zf:
	for i in range(n_sample_check):
		with zf.open(f'sample_{i:06d}.pt') as f:
			data = torch.load(io.BytesIO(f.read()))
			feild_32.append(data['image'])

feild_32 = torch.concat(feild_32, dim=0).numpy()
print(feild_32.shape)

# tau  = 1 
field_corr_true = []
# feild_32 = 
for tau in range(1, 50):
	v_cor = np.mean(feild_32[tau:, ...] * feild_32[:-tau, ...]) / 
	field_corr_true.append(np.mean(feild_32[tau:, ...] * feild_32[:-tau, ...]) )

plt.plot(field_corr_true)
plt.show()

In [ ]:
plt.plot(field_corr_true, label='true')
plt.plot(field_corr_gen, label='gen')
plt.legend()
plt.show()